# 00 · Pilot: 20 raw samples + timing one training epoch on a T4

What this notebook does, top to bottom (Runtime → Run all, about 5–10 minutes):

1. Pulls a small pilot set (about 40 images per class, ~200 total, about 10% of the planned data) straight from the **ISIC Archive API**, dermoscopic images only.
2. Saves **20 raw samples** (4 per class, original files untouched) with their metadata, licenses and attribution → `raw_samples.zip`, plus a copy in your Google Drive (private, as the proposal says).
3. Resizes the pilot images **once** to 224×224 (spec rule 2), audits the metadata, and splits by patient so no patient lands in two splits.
4. Times one training epoch of the planned model (ImageNet-pretrained **ResNet-18, 5-output head**) on the GPU, frozen backbone and with the last block unfrozen, and prints the full-run estimate to paste into the proposal.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Classes: melanoma, basal cell carcinoma (bcc), squamous cell carcinoma (scc), benign melanocytic nevus, benign keratosis. Actinic keratosis and uncertain diagnoses are excluded, as the proposal says.

In [ ]:
import io, re, time, random, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "<- switch the runtime to a T4 GPU")

## Settings
`PLANNED_TOTAL` is the top of the proposal's 1,500–2,500 target, so the full-run estimate is conservative.

In [ ]:
API = "https://api.isic-archive.com/api/v2/"
CLASSES = ["melanoma", "bcc", "scc", "nevus", "benign_keratosis"]
PILOT_PER_CLASS = 40        # 5 x 40 = 200 pilot images (about 10% of the plan)
RAW_PER_CLASS = 4           # 5 x 4 = the 20 raw samples
PLANNED_TOTAL = 2500        # top of the proposal's 1,500-2,500 range
PLANNED_EPOCHS = 10
IMG_SIZE, BATCH_SIZE = 224, 32
WORK = Path("/content/pilot")
DRIVE_DIR = "CSC451_project"  # private Google Drive folder for the raw samples

# Searches to page through (ISIC search syntax). The last one is a fallback if the
# diagnosis_1 field ever changes; every record is still mapped by the rules below.
SCAN_QUERIES = ["diagnosis_1:Malignant AND image_type:dermoscopic",
                "diagnosis_1:Benign AND image_type:dermoscopic",
                "image_type:dermoscopic"]
MAX_PAGES_PER_QUERY = 80    # 100 records per page; metadata only, fast

# Label mapping: first matching rule wins. The team's 100-record agreement check reviews these.
EXCLUDE = r"actinic|indeterminate|uncertain|atypical|collision"
CLASS_RULES = [("melanoma", r"melanoma"),
               ("bcc", r"basal cell"),
               ("scc", r"squamous cell"),
               ("nevus", r"\bnev(us|i)\b"),
               ("benign_keratosis", r"seborrh|solar lentigo|lichen planus|benign keratosis")]

## 1 · Find pilot images in the ISIC Archive (metadata only)

In [ ]:
session = requests.Session()

def get_page(url, params=None):
    r = session.get(url, params=params, timeout=90)
    r.raise_for_status()
    return r.json()

def flatten(rec):
    md = rec.get("metadata", {})
    full = rec.get("files", {}).get("full", {})
    return {"isic_id": rec["isic_id"], "license": rec.get("copyright_license"),
            "attribution": rec.get("attribution"), "url": full.get("url"), "bytes": full.get("size"),
            **md.get("acquisition", {}), **md.get("clinical", {})}

def diagnosis_text(row):
    """Most specific diagnosis available, lower-cased."""
    for c in ("diagnosis_3", "diagnosis_2", "diagnosis"):
        v = row.get(c)
        if isinstance(v, str) and v.strip():
            return v.strip().lower()
    return ""

def map_label(row):
    text = diagnosis_text(row)
    if not text or re.search(EXCLUDE, text):
        return None
    for cls, pattern in CLASS_RULES:
        if re.search(pattern, text):
            return cls
    return None

picked = {c: [] for c in CLASSES}
seen, scanned = set(), []
for q in SCAN_QUERIES:
    if all(len(v) >= PILOT_PER_CLASS for v in picked.values()):
        break
    try:
        page = get_page(API + "images/search/", {"query": q, "limit": 100})
    except requests.HTTPError as e:  # e.g. a field name the archive no longer uses
        print("skipped (search failed):", q, "-", e)
        continue
    print(f"{page['count']:>7,} archive matches  <- {q}")
    for _ in range(MAX_PAGES_PER_QUERY):
        for rec in page["results"]:
            row = flatten(rec)
            scanned.append(row)
            cls = map_label(row)
            if cls and row["isic_id"] not in seen and row["url"] and len(picked[cls]) < PILOT_PER_CLASS:
                seen.add(row["isic_id"]); picked[cls].append({**row, "label": cls})
        if not page["next"] or all(len(v) >= PILOT_PER_CLASS for v in picked.values()):
            break
        page = get_page(page["next"])

scanned = pd.DataFrame(scanned)
meta = pd.DataFrame([r for v in picked.values() for r in v])
print("\nPilot images per class:", meta.label.value_counts().reindex(CLASSES, fill_value=0).to_dict())
short = [c for c in CLASSES if (meta.label == c).sum() < PILOT_PER_CLASS]
if short:
    print("WARNING: fewer than", PILOT_PER_CLASS, "found for", short, "- raise MAX_PAGES_PER_QUERY or check the mapping rules")

# Exact diagnosis labels seen in the archive: use these for the data README and the label-mapping check.
diag_cols = [c for c in ("diagnosis_1", "diagnosis_2", "diagnosis_3", "diagnosis") if c in scanned]
print(f"\nScanned {len(scanned):,} records. Most common diagnosis labels:")
if diag_cols:
    print(scanned.groupby(diag_cols, dropna=False).size().sort_values(ascending=False).head(25).to_string())

## 2 · Download, keep the 20 raw samples, resize once to 224×224

In [ ]:
RAW_DIR, IMG_DIR = WORK / "raw_samples", WORK / f"img{IMG_SIZE}"
for d in (RAW_DIR, IMG_DIR):
    d.mkdir(parents=True, exist_ok=True)
keep_raw = set(meta.groupby("label").head(RAW_PER_CLASS).isic_id)

def fetch(row):
    try:
        data = requests.get(row.url, timeout=180).content
        if row.isic_id in keep_raw:  # untouched original file
            (RAW_DIR / row.label).mkdir(exist_ok=True)
            (RAW_DIR / row.label / f"{row.isic_id}.jpg").write_bytes(data)
        img = Image.open(io.BytesIO(data)).convert("RGB")
        w, h = img.size
        # center-crop to square, then resize: keeps the (centered) lesion undistorted
        ImageOps.fit(img, (IMG_SIZE, IMG_SIZE)).save(IMG_DIR / f"{row.isic_id}.jpg", quality=95)
        return w, h, None
    except Exception as e:  # corrupt or unreachable file: counted, not fatal
        return None, None, repr(e)

t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    results = list(ex.map(fetch, meta.itertuples()))
meta["width"], meta["height"], meta["error"] = zip(*results)
print(f"Downloaded {len(meta)} images in {time.time() - t0:.0f}s; failed: {meta.error.notna().sum()}")
meta = meta[meta.error.isna()].drop(columns="error").reset_index(drop=True)
meta.drop(columns=["url"]).to_csv(WORK / "pilot_metadata.csv", index=False)

In [ ]:
# The 20 raw samples: original files + metadata + licenses + attribution, zipped.
samples = meta[meta.isic_id.isin(keep_raw)].drop(columns=["url"])
samples.to_csv(RAW_DIR / "metadata.csv", index=False)
(RAW_DIR / "attribution.txt").write_text("\n".join(sorted(samples.attribution.dropna().unique())))
(RAW_DIR / "README.txt").write_text(
    "20 raw ISIC Archive samples (4 per class), original files as downloaded on "
    f"{time.strftime('%Y-%m-%d')} via https://api.isic-archive.com/api/v2/. "
    "License and attribution per image are in metadata.csv.\n")
zip_path = shutil.make_archive("/content/raw_samples", "zip", RAW_DIR)
print(samples.groupby("label").size().to_dict(), "->", zip_path)

try:  # private copy in Google Drive + browser download
    from google.colab import drive, files
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive") / DRIVE_DIR
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_path, dest / "raw_samples.zip")
    print("Copied to Drive:", dest / "raw_samples.zip")
    files.download(zip_path)
except Exception as e:
    print("Not in Colab (or Drive not mounted):", e)

## 3 · Quick metadata audit (numbers for the data section)

In [ ]:
print("Licenses:", meta.license.value_counts(dropna=False).to_dict())
print(f"Raw sizes: {meta.width.min()}x{meta.height.min()} to {meta.width.max()}x{meta.height.max()} px")
for c in ["patient_id", "lesion_id", "diagnosis_confirm_type", "fitzpatrick_skin_type", "anatom_site_general"]:
    if c in meta:
        print(f"{c:24s} present {meta[c].notna().sum():3d}/{len(meta)}  unique {meta[c].nunique()}")
    else:
        print(f"{c:24s} not in this pilot's metadata")

## 4 · Split by patient (70/15/15) — no patient in two splits

In [ ]:
group = meta["patient_id"] if "patient_id" in meta else pd.Series(np.nan, index=meta.index, dtype=object)
if "lesion_id" in meta:
    group = group.fillna(meta["lesion_id"])   # no patient id: fall back to the lesion
meta["group"] = group.fillna(meta["isic_id"])  # no id at all: the image is its own group

train_i, rest_i = next(GroupShuffleSplit(1, test_size=0.30, random_state=SEED).split(meta, groups=meta.group))
rest = meta.iloc[rest_i]
val_j, test_j = next(GroupShuffleSplit(1, test_size=0.50, random_state=SEED).split(rest, groups=rest.group))
meta["split"] = "train"
meta.loc[rest.index[val_j], "split"] = "val"
meta.loc[rest.index[test_j], "split"] = "test"
print(pd.crosstab(meta.label, meta.split).reindex(CLASSES))
train_groups = set(meta.group[meta.split == "train"])
assert not train_groups & set(meta.group[meta.split != "train"]), "a patient/lesion leaked across splits"
print("OK: no patient or lesion appears in two splits")

## 5 · Planned model and one timed epoch

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet statistics
train_tf = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

class Lesions(Dataset):
    def __init__(self, df, tf):
        self.paths = [IMG_DIR / f"{i}.jpg" for i in df.isic_id]
        self.y = [CLASS_TO_IDX[c] for c in df.label]
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.y[i]

pin = device == "cuda"
train_dl = DataLoader(Lesions(meta[meta.split == "train"], train_tf), BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=pin)
val_dl = DataLoader(Lesions(meta[meta.split == "val"], eval_tf), BATCH_SIZE, num_workers=2, pin_memory=pin)

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))      # 5-output head
model = model.to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-18, {n_params:,} parameters, input 3x{IMG_SIZE}x{IMG_SIZE}")
loss_fn = nn.CrossEntropyLoss()

In [ ]:
def set_trainable(unfreeze_last_block):
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith("fc.") or (unfreeze_last_block and name.startswith("layer4."))
    return torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)

def run_epoch(dl, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
                loss = loss_fn(model(x), y)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total += loss.item() * len(y); n += len(y)
    return total / n

def timed(fn, *args):
    if device == "cuda": torch.cuda.synchronize()
    t = time.time(); out = fn(*args)
    if device == "cuda": torch.cuda.synchronize()
    return out, time.time() - t

scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
_, warmup_s = timed(run_epoch, train_dl, set_trainable(False))   # first epoch pays CUDA start-up cost

times = {}
for label, unfreeze in [("frozen backbone", False), ("last block unfrozen", True)]:
    opt = set_trainable(unfreeze)
    _, train_s = timed(run_epoch, train_dl, opt)
    _, val_s = timed(run_epoch, val_dl)
    times[label] = (train_s, val_s)
    print(f"{label:20s} train epoch {train_s:6.1f}s   validation {val_s:5.1f}s")

## 6 · Full-run estimate (paste into the proposal)
Proposal formula: (t × full training examples / n × 10) / 60, here with validation included and using the slower configuration.

In [ ]:
n_pilot = int((meta.split == "train").sum())
n_full = int(PLANNED_TOTAL * 0.70)
label, (train_s, val_s) = max(times.items(), key=lambda kv: sum(kv[1]))
epoch_s = train_s + val_s
full_min = epoch_s * (n_full / n_pilot) * PLANNED_EPOCHS / 60
gpu = torch.cuda.get_device_name(0) if device == "cuda" else "CPU (not a valid pilot: use a T4)"

summary = f"""PILOT TIMING ({time.strftime('%Y-%m-%d')}, {gpu})
Model: ImageNet-pretrained ResNet-18, 5-output head, {n_params:,} parameters, 3x{IMG_SIZE}x{IMG_SIZE} input
Pilot examples: {len(meta)} dermoscopic ISIC images ({n_pilot} in the training split)
Epoch time ({label}, slower setting): train {train_s:.1f}s + validation {val_s:.1f}s = {epoch_s:.1f}s
Frozen backbone: {sum(times['frozen backbone']):.1f}s per epoch; first warm-up epoch {warmup_s:.1f}s (start-up cost, excluded)
Planned epochs: {PLANNED_EPOCHS}
Full-run estimate: {epoch_s:.1f}s x {n_full}/{n_pilot} x {PLANNED_EPOCHS} / 60 = {full_min:.1f} min (+ checkpoint overhead)
Within the 30-minute limit: {'YES' if full_min <= 30 else 'NO - shrink images, model, or data'}"""
print(summary)
(WORK / "pilot_summary.txt").write_text(summary)
try:
    from google.colab import files
    files.download(str(WORK / "pilot_summary.txt"))
except Exception:
    pass